# CXR-XAI-Clinical — Kaggle Training

**Before running:**
1. Add dataset → search **NIH Chest X-rays** (owner: nih-chest-xrays) → Add
2. Set accelerator → **GPU T4 x2** (P100 is incompatible with modern PyTorch)
3. Set your W&B API key in *Add-ons → Secrets* as `WANDB_API_KEY`

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os, subprocess

REPO = "https://github.com/saeed-amini/cxr-xai-clinical.git"  # update to your repo URL
REPO_DIR = "/kaggle/working/cxr-xai-clinical"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# ── 2. Install dependencies (torch/torchvision already on Kaggle) ─────────────
!pip install -q timm grad-cam captum quantus wandb pyyaml tqdm scikit-learn scipy pydicom seaborn

In [ ]:
# ── 3. Flatten images: symlink all images_00x/images/* into one directory ─────
# The NIH dataset on Kaggle splits 112k images across 12 sub-folders.
# We create symlinks in /kaggle/working/nih14_images so the dataset class
# sees a single flat directory (read-only input is preserved).
from pathlib import Path

src_base = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
img_dir  = Path("/kaggle/working/nih14_images")
img_dir.mkdir(exist_ok=True)

created = 0
for subdir in sorted(src_base.glob("images_*/images")):
    for img in subdir.iterdir():
        link = img_dir / img.name
        if not link.exists():
            link.symlink_to(img)
            created += 1

total = sum(1 for _ in img_dir.iterdir())
print(f"Symlinks created: {created}  |  Total images: {total}")

In [ ]:
# ── 4. Verify GPU + data paths ────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

from pathlib import Path
checks = {
    "images dir":       Path("/kaggle/working/nih14_images"),
    "Data_Entry_2017":  Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv"),
    "train_val_list":   Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data/train_val_list.txt"),
    "test_list":        Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data/test_list.txt"),
}
for label, p in checks.items():
    print(f"  [{'OK    ' if p.exists() else 'MISSING'}] {label}: {p}")

In [ ]:
# ── 5. W&B login ──────────────────────────────────────────────────────────────
import wandb, os
wandb.login(key=os.environ.get("WANDB_API_KEY"))

In [ ]:
# ── 6a. Train DenseNet-121 ────────────────────────────────────────────────────
!python scripts/train.py --config configs/train_kaggle.yaml --model densenet121

In [ ]:
# ── 6b. Train ViT-Base/16  (if OOM: set batch_size: 16 in train_kaggle.yaml) ─
!python scripts/train.py --config configs/train_kaggle.yaml --model vit_base_patch16_224

In [ ]:
# ── 7. Generate XAI maps ──────────────────────────────────────────────────────
!python scripts/generate_xai.py --config configs/train_kaggle.yaml --model densenet121
!python scripts/generate_xai.py --config configs/train_kaggle.yaml --model vit_base_patch16_224

In [ ]:
# ── 8. Evaluate quantitative XAI metrics ─────────────────────────────────────
!python scripts/evaluate.py --config configs/train_kaggle.yaml --model densenet121
!python scripts/evaluate.py --config configs/train_kaggle.yaml --model vit_base_patch16_224

In [ ]:
# ── 9. List outputs ───────────────────────────────────────────────────────────
import os
for root, dirs, files in os.walk("/kaggle/working/results"):
    for f in files:
        print(os.path.join(root, f))